[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/retraction_mode_playground.ipynb)

# MarinFold — retraction-mode contacts-v1 (#175)

Interactive playground for **`exp175-cv1-1.5B-mode50-v2`**, a contacts-v1 model that can
**take back its own predictions mid-rollout** with a `<retract>` statement — and that lets you
choose, at token 0, whether it is allowed to.

It was fine-tuned from `contacts-v1-exp120-1.5B` on a 50:50 mix of self-correction traces
([#159](https://github.com/Open-Athena/MarinFold/issues/159)) and ordinary contacts-v1 documents.

**Two modes, one checkpoint** — the first token decides:

| prompt token 0 | behaviour | retracts / rollout |
|---|---|---|
| `<contacts-v1>` | ordinary contacts-v1 | **0.1** |
| `<contacts-v1.backtracking>` | may retract | **42.0** |

### Read this before drawing conclusions

On the 554-protein #89 benchmark this model is **slightly worse at contact prediction** than the
`exp120` base it was fine-tuned from — R-precision −0.006 in clean mode, −0.015 in retraction
mode. It is published because the *mechanism* is interesting, not because it is a good contact
predictor. For accuracy, use `marinfold infer --model 1.5B`.

Runs on a free Colab T4. No login needed — the bucket reads anonymously.


In [ ]:
#@title Install (~2 min)
!pip -q install 'transformers>=4.44,<5' 'huggingface_hub>=1.5' accelerate safetensors

# NOTE: pinned to the exp175 branch until PR #197 lands on main. The mode
# switch is GenerationConfig(backtracking=True), which main does not have yet;
# installing from main gives
#     TypeError: GenerationConfig.__init__() got an unexpected keyword
#     argument 'backtracking'
# After the merge, drop the '@exp175-backtracking-mode-token'.
!pip -q install 'marinfold @ git+https://github.com/Open-Athena/MarinFold.git@exp175-backtracking-mode-token#subdirectory=marinfold'
print('ready')


In [ ]:
#@title Download the model (~2.8 GB, anonymous)
from pathlib import Path
from huggingface_hub import BucketFile, download_bucket_files, list_bucket_tree

BUCKET = 'open-athena/MarinFold'
PREFIX = 'checkpoints/exp175-cv1-1_5b-mode50-v2-lr3e-4-e1-cos/hf/step-2070'
MODEL_DIR = Path('/content/exp175'); MODEL_DIR.mkdir(parents=True, exist_ok=True)

# token=False everywhere: the bucket is public, and passing no token avoids a
# stale cached credential 401ing. Buckets are their own storage layer -- the
# usual snapshot_download does NOT see bucket contents -- so list and download
# through the bucket API instead.
files = [f for f in list_bucket_tree(BUCKET, prefix=PREFIX + '/', recursive=True, token=False)
         if isinstance(f, BucketFile)]
assert files, f'nothing under {PREFIX} -- check the path'
download_bucket_files(
    BUCKET,
    [(f, MODEL_DIR / Path(f.path).name) for f in files],
    raise_on_missing_files=True, token=False)
print(sorted(p.name for p in MODEL_DIR.iterdir()))


In [ ]:
#@title Load
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, dtype=torch.bfloat16,
    device_map='cuda' if torch.cuda.is_available() else 'cpu')
model.eval()

# Expect a warning that rope_scaling's original_max_position_embeddings must be
# less than max_position_embeddings (both are 8192 here). It is a validation
# complaint, not a failure: the llama3 scaling is still applied -- checked by
# comparing inv_freq against the unscaled rope, which differs by 8x at the low
# frequencies. The eval that produced the numbers above ran on this same config.

# The whole experiment rests on these ids existing.
for t in ('<contacts-v1>', '<contacts-v1.backtracking>', '<retract>', '<end>'):
    i = tok.convert_tokens_to_ids(t)
    assert i != tok.unk_token_id, f'{t} is UNK -- wrong tokenizer'
    print(f'{t:30s} {i}')


## Predict contacts for a sequence

A contacts-v1 document is a **sequence section** (residues at randomised position tokens)
followed by a **structure section** of `<contact> <pX> <pY>` statements. In retraction mode the
structure section is an ordered *edit list*: `<retract> <pX> <pY>` takes a pair back, and the
prediction is whatever is still live at `<end>`.


In [ ]:
#@title Your protein  { run: 'auto' }
SEQUENCE = 'MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKR' #@param {type:'string'}
MODE = 'backtracking' #@param ['backtracking', 'clean']
N_ROLLOUTS = 20 #@param {type:'integer'}
TEMPERATURE = 1.0 #@param {type:'number'}

from marinfold.document_structures.contacts_v1 import (
    GenerationConfig, build_document, residues_from_sequence)

NUM_POS, MIN_SEP, BEGIN = 2000, 6, '<begin_statements>'
residues = residues_from_sequence(SEQUENCE)
L = len(residues)

def make_prompt(mode, k):
    """One rollout's prompt + its position->residue map."""
    cfg = GenerationConfig(backtracking=(mode == 'backtracking'))
    # The entry id seeds the position permutation, so varying it per rollout
    # gives each one a different residue<->position assignment -- that is the
    # recipe the benchmark uses, not an incidental detail.
    d = build_document(f'demo:r{k}', residues, [], config=cfg)
    prompt = d.document[: d.document.index(BEGIN) + len(BEGIN)]
    seqidx = {(d.n_term_index + t) % NUM_POS: t for t in range(d.seq_len)}
    return prompt, seqidx

print(f'L={L}   token 0 = ' + make_prompt(MODE, 0)[0].split()[0])


In [ ]:
#@title Generate rollouts
import torch

end_id = tok.convert_tokens_to_ids('<end>')

def rollout(mode, n, temperature=1.0):
    """Return [(text, seqidx)] -- n sampled structure sections."""
    out = []
    for k in range(n):
        prompt, seqidx = make_prompt(mode, k)
        ids = tok(prompt, return_tensors='pt').input_ids.to(model.device)
        budget = min(8192 - ids.shape[1], 8 * L + 128)
        with torch.no_grad():
            o = model.generate(ids, do_sample=True, temperature=temperature,
                               top_p=0.95, max_new_tokens=budget,
                               eos_token_id=end_id, pad_token_id=tok.pad_token_id)
        out.append((tok.decode(o[0, ids.shape[1]:], skip_special_tokens=False), seqidx))
        print(f'  rollout {k + 1}/{n}', end='\r')
    return out

outs = rollout(MODE, N_ROLLOUTS, TEMPERATURE)
print(f'\n{len(outs)} rollouts')


In [ ]:
#@title What did it retract?
from marinfold.document_structures.contacts_v1.read import (
    fold_statements, iter_structure_statements)

stmts = list(iter_structure_statements(outs[0][0]))
fold = fold_statements(stmts)
print(f'{fold.n_contact} contacts asserted, {fold.n_retract} retracted -> {len(fold.live)} live at <end>')

# Each retraction with how far back it reached. The training traces place
# retractions ~18 statements behind the mistake; a model that had learned the
# token but not the behaviour would only ever undo its own last statement.
emitted = {}
for n, (kind, a, b) in enumerate(stmts):
    pair = (min(a, b), max(a, b))
    if kind == 'contact':
        emitted[pair] = n
    else:
        at = emitted.get(pair)
        back = 'never emitted' if at is None else f'{n - at} statements back'
        print(f'  stmt {n:4d}  <retract> <p{a}> <p{b}>   ({back})')


## Vote across rollouts into a contact map

Single rollouts are noisy; the settled recipe
([#82](https://github.com/Open-Athena/MarinFold/issues/82)) samples many and votes. Note the fold
runs **before** voting — a pair the rollout took back must not vote for itself.


In [ ]:
#@title Contact map
import numpy as np, matplotlib.pyplot as plt

def vote_map(rollouts):
    votes = np.zeros((L, L))
    for text, seqidx in rollouts:
        # Fold FIRST: a pair the rollout took back must not vote for itself.
        for pa, pb in fold_statements(iter_structure_statements(text)).live:
            ia, ib = seqidx.get(pa), seqidx.get(pb)
            if ia is None or ib is None or abs(ia - ib) < MIN_SEP:
                continue
            votes[min(ia, ib), max(ia, ib)] += 1
    return (votes + votes.T) / max(len(rollouts), 1)

plt.figure(figsize=(6, 5.5))
plt.imshow(vote_map(outs), cmap='viridis', origin='lower')
plt.colorbar(label=f'fraction of {len(outs)} rollouts')
plt.xlabel('residue'); plt.ylabel('residue')
plt.title(f'{MODE} mode -- predicted contacts'); plt.tight_layout(); plt.show()


## Compare the two modes side by side

The same weights, the same protein, one token different. This is the comparison the mode marker
exists to make possible — before it, the model had no way to be told which mode it was in, and
hedged across both.


In [ ]:
#@title Both modes
for mode in ('clean', 'backtracking'):
    rs = rollout(mode, 10)
    folds = [fold_statements(iter_structure_statements(t)) for t, _ in rs]
    print(f'{mode:14s} contacts/rollout {np.mean([f.n_contact for f in folds]):6.1f}   '
          f'retracts/rollout {np.mean([f.n_retract for f in folds]):5.1f}   '
          f'live at <end> {np.mean([len(f.live) for f in folds]):6.1f}')


## Where this came from

- [#158](https://github.com/Open-Athena/MarinFold/issues/158) — the `<retract>` statement and its fold
- [#159](https://github.com/Open-Athena/MarinFold/issues/159) — the self-correction corpus (and the
  sorted-flush bug that invalidated the first two training runs)
- [#160](https://github.com/Open-Athena/MarinFold/issues/160) — first training run
- [#175](https://github.com/Open-Athena/MarinFold/issues/175) — the mode marker and this model

**Open question this model does not answer:** its retraction captures only ~42% of the available
discrimination (`P(FP | retracted)` = 0.888 against a 0.774 base rate), while the corpus that
taught it reaches 97%. Removing the corpus's ordering artifact did not move that number at all,
so the ceiling is something else — a good thing to poke at here.
